# NB31 · Real-World Programs

`advanced/` · 20 questions · the last one is a story project

How real programs are put together: command-line tools, configuration,
small databases, other programs run as subprocesses, archives and package
layout. Every task is a small real job (a to-do CLI, a shop database, a
log summary, a backup). Command-line tools are driven by calling
`parse_args([...])` with a list of arguments, databases live in memory
(`":memory:"`), files live in temporary folders, and subprocesses only
run `sys.executable`, so everything is safe to run anywhere.

## You'll need

- NB13: files
- NB28: logging

## You'll practise

- Project layout, packages, __main__.py
- argparse (called with an explicit argument list)
- Config from os.environ and tomllib (3.11+)
- sqlite3: in-memory databases, parameterised queries, transactions, row_factory
- subprocess.run with sys.executable
- shutil and zipfile
- venv, pip and pyproject.toml explained
- Template strings (t-strings, 3.14) for safe interpolation

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

A greeting tool takes `--name` (text) and `--times` (a whole number,
default 1). Build the parser, parse the given argument list, and print the
greeting `Hello, <name>!` that many times.

**Expected output:**

```text
Hello, Ana!
Hello, Ana!
```

<details><summary><b>Solution</b></summary>

```python
import argparse
parser = argparse.ArgumentParser()
parser.add_argument("--name")
parser.add_argument("--times", type=int, default=1)
args = parser.parse_args(argv)
for _ in range(args.times):
    print("Hello, " + args.name + "!")
```

</details>

In [ ]:
argv = ["--name", "Ana", "--times", "2"]

# your code here


### Q02 · Real-life

Open a fresh in-memory database, create a table `pets` with a text `name`
and an integer `age`, insert one pet (`Rex`, 4) using placeholders, and
print the rows you get back with a `SELECT`.

**Expected output:**

```text
[('Rex', 4)]
```

<details><summary><b>Solution</b></summary>

```python
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE pets (name TEXT, age INTEGER)")
conn.execute("INSERT INTO pets VALUES (?, ?)", ("Rex", 4))
print(conn.execute("SELECT * FROM pets").fetchall())
```

</details>

In [ ]:
# your code here


### Q03 · Real-life

An app's settings arrive as TOML text. Parse it and print the database
host and the debug flag.

**Expected output:**

```text
db.local
True
```

<details><summary><b>Solution</b></summary>

```python
import tomllib
settings = tomllib.loads(settings_text)
print(settings["database"]["host"])
print(settings["server"]["debug"])
```

</details>

In [ ]:
settings_text = """
[server]
port = 8080
debug = true

[database]
host = "db.local"
"""

# your code here


### Q04 · Real-life

Read the setting `APP_PORT` from the environment as a number. Then read
`APP_MODE`, falling back to `"production"` when it isn't set. Print both.

**Expected output:**

```text
8080
production
```

<details><summary><b>Solution</b></summary>

```python
port = int(os.environ["APP_PORT"])
mode = os.environ.get("APP_MODE", "production")
print(port)
print(mode)
```

</details>

In [ ]:
import os
os.environ["APP_PORT"] = "8080"
os.environ.pop("APP_MODE", None)

# your code here


### Q05 · Output-match

Run a child Python process (using the same interpreter that runs this
code) that prints the result of `6 * 7`. Capture its output as text and
print the output without the trailing newline, then print its return code.

**Expected output:**

```text
42
0
```

<details><summary><b>Solution</b></summary>

```python
import subprocess, sys
result = subprocess.run([sys.executable, "-c", "print(6 * 7)"],
                        capture_output=True, text=True)
print(result.stdout.strip())
print(result.returncode)
```

</details>

In [ ]:
# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A user types their name into a search box and the program looks them up.
Look up the user whose name is **exactly** `typed`, in a way that cannot
be tricked by what the user types. Print how many rows matched, then
prove the lookup still works by searching for `"Bea"` and printing that
row.

**Expected output:**

```text
0
[('Bea', 'guest')]
```

<details><summary><b>Solution</b></summary>

```python
query = "SELECT * FROM users WHERE name = ?"
print(len(conn.execute(query, (typed,)).fetchall()))
print(conn.execute(query, ("Bea",)).fetchall())
```

</details>

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (name TEXT, role TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)",
                 [("Ana", "admin"), ("Bea", "guest"), ("Cy", "guest")])
typed = "x' OR '1'='1"

# your code here


### Q07 · Build

A to-do tool has two sub-commands: `add TEXT` (with an optional
`--priority`, a number, default 3) and `done ID` (ID is a number). Build
the parser with sub-commands and parse each argument list in `commands`.
For each, print the command name and then its values, as in the expected
output.

**Expected output:**

```text
add buy milk 1
add call mum 3
done 7
```

<details><summary><b>Solution</b></summary>

```python
import argparse
parser = argparse.ArgumentParser(prog="todo")
sub = parser.add_subparsers(dest="command", required=True)
add = sub.add_parser("add")
add.add_argument("text")
add.add_argument("--priority", type=int, default=3)
done = sub.add_parser("done")
done.add_argument("id", type=int)
for argv in commands:
    args = parser.parse_args(argv)
    if args.command == "add":
        print("add", args.text, args.priority)
    else:
        print("done", args.id)
```

</details>

In [ ]:
commands = [["add", "buy milk", "--priority", "1"], ["add", "call mum"], ["done", "7"]]

# your code here


### Q08 · Real-life

Query the `books` table so that each row can be read **by column name**
instead of by position. Print each book as `title by author`, ordered by
title.

**Expected output:**

```text
Dune by Herbert
Emma by Austen
Ulysses by Joyce
```

<details><summary><b>Solution</b></summary>

```python
conn.row_factory = sqlite3.Row
for row in conn.execute("SELECT * FROM books ORDER BY title"):
    print(row["title"], "by", row["author"])
```

</details>

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE books (title TEXT, author TEXT, year INTEGER)")
conn.executemany("INSERT INTO books VALUES (?, ?, ?)",
                 [("Dune", "Herbert", 1965), ("Emma", "Austen", 1815), ("Ulysses", "Joyce", 1922)])

# your code here


### Q09 · Real-life

Create three small text files in a temporary folder, pack them into one
zip archive (also inside that folder), then open the archive and print
its file names sorted. Finally print the contents of `b.txt` read
straight from the archive.

**Expected output:**

```text
['a.txt', 'b.txt', 'c.txt']
data in b.txt
```

<details><summary><b>Solution</b></summary>

```python
import zipfile
with tempfile.TemporaryDirectory() as folder:
    for name in names:
        Path(folder, name).write_text("data in " + name)
    archive_path = Path(folder, "bundle.zip")
    with zipfile.ZipFile(archive_path, "w") as archive:
        for name in names:
            archive.write(Path(folder, name), arcname=name)
    with zipfile.ZipFile(archive_path) as archive:
        print(sorted(archive.namelist()))
        print(archive.read("b.txt").decode())
```

</details>

In [ ]:
import tempfile
from pathlib import Path
names = ["a.txt", "b.txt", "c.txt"]

# your code here


### Q10 · Puzzle

A search page puts the user's text into HTML, and the text may contain
`<` or `&`. A **t-string** (a string with a `t` in front, Python 3.14)
keeps the values separate from the fixed text. Write `safe_html(template)`
that joins the fixed text as it is, but runs every interpolated value
through `html.escape` (as text). Use it on a t-string with the name
`visitor` and print the result.

**Expected output:**

```text
<p>Hello &lt;b&gt;Eve &amp; Co&lt;/b&gt;</p>
```

<details><summary><b>Solution</b></summary>

```python
import html
from string.templatelib import Interpolation
def safe_html(template):
    parts = []
    for part in template:
        if isinstance(part, Interpolation):
            parts.append(html.escape(str(part.value)))
        else:
            parts.append(part)
    return "".join(parts)
print(safe_html(t"<p>Hello {visitor}</p>"))
```

</details>

In [ ]:
visitor = "<b>Eve & Co</b>"

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

Settings are layered: built-in defaults, then a TOML file, then
environment variables, each layer overriding the one before it. Merge
the layers (`env` already holds only the relevant values, with numbers as
text) and print the final settings as a dict sorted by key.

**Expected output:**

```text
{'debug': False, 'name': 'shop', 'port': 9000}
```

<details><summary><b>Solution</b></summary>

```python
from_file = tomllib.loads(file_text)
from_env = {}
if "APP_PORT" in env:
    from_env["port"] = int(env["APP_PORT"])
settings = defaults | from_file | from_env
print(dict(sorted(settings.items())))
```

</details>

In [ ]:
import tomllib
defaults = {"port": 80, "debug": False, "name": "app"}
file_text = "port = 8080\nname = \"shop\""
env = {"APP_PORT": "9000"}

# your code here


### Q12 · Real-life

A bank transfer must move money **all or nothing**. Move `amount` from
one account to the other inside one transaction (`with conn:`). The table
refuses negative balances, so a transfer that is too large fails: catch
the database's integrity error and print `Transfer failed`. After every
transfer, successful or not, print both balances (as a list of tuples,
ordered by name). Do this for both transfers in `amounts`, moving money
from `ana` to `ben`.

**Expected output:**

```text
[('ana', 70), ('ben', 50)]
Transfer failed
[('ana', 70), ('ben', 50)]
```

<details><summary><b>Solution</b></summary>

```python
for amount in amounts:
    try:
        with conn:
            conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, "ben"))
            conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, "ana"))
    except sqlite3.IntegrityError:
        print("Transfer failed")
    print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())
```

</details>

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ana", 100), ("ben", 20)])
conn.commit()
amounts = [30, 500]

# your code here


### Q13 · Build

Create a tiny package in a temporary folder: a folder `greeter` holding an
empty `__init__.py` and a `__main__.py` that prints `Hello from greeter`
followed by any command-line arguments. Then run the package as a program
(`-m greeter`, with the interpreter that runs this code, from inside the
temporary folder) with the extra argument `Ana`, and print what it
printed.

**Expected output:**

```text
Hello from greeter Ana
```

<details><summary><b>Solution</b></summary>

```python
with tempfile.TemporaryDirectory() as folder:
    package = Path(folder, "greeter")
    package.mkdir()
    (package / "__init__.py").write_text("")
    (package / "__main__.py").write_text(
        "import sys\nprint('Hello from greeter', *sys.argv[1:])\n")
    result = subprocess.run([sys.executable, "-m", "greeter", "Ana"],
                            cwd=folder, capture_output=True, text=True)
    print(result.stdout.strip())
```

</details>

In [ ]:
import subprocess, sys, tempfile
from pathlib import Path

# your code here


### Q14 · Real-life

Run two child programs and treat their arguments safely: pass each
argument as a separate list item, never glue a shell command together.
The first child prints its own first argument back (use the nasty text
`file; echo hacked` as that argument), the second exits with status 3.
For the second, use `check=True`, catch the error and print its return
code.

**Expected output:**

```text
file; echo hacked
child failed with 3
```

<details><summary><b>Solution</b></summary>

```python
nasty = "file; echo hacked"
echo = subprocess.run(
    [sys.executable, "-c", "import sys; print(sys.argv[1])", nasty],
    capture_output=True, text=True)
print(echo.stdout.strip())
try:
    subprocess.run([sys.executable, "-c", "raise SystemExit(3)"], check=True)
except subprocess.CalledProcessError as error:
    print("child failed with", error.returncode)
```

</details>

In [ ]:
import subprocess, sys

# your code here


### Q15 · Real-life

The project's `pyproject.toml` is given as text. Parse it and print the
project name and version on one line (`name version`), then the
dependencies sorted alphabetically (one per line), then the command that
the `[project.scripts]` table installs, as `command -> target`.

**Expected output:**

```text
shopkit 0.3.1
attrs
click
requests>=2.30
shop -> shopkit.cli:main
```

<details><summary><b>Solution</b></summary>

```python
import tomllib
data = tomllib.loads(pyproject)
project = data["project"]
print(project["name"], project["version"])
for dependency in sorted(project["dependencies"]):
    print(dependency)
for command, target in project["scripts"].items():
    print(command, "->", target)
```

</details>

In [ ]:
pyproject = """
[project]
name = "shopkit"
version = "0.3.1"
requires-python = ">=3.10"
dependencies = ["requests>=2.30", "click", "attrs"]

[project.scripts]
shop = "shopkit.cli:main"
"""

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Real-life

Write `main(argv)` for a log summary tool. It takes a log file path, an
optional `--level` (only `INFO`, `WARN` or `ERROR`, default `ERROR`) and
an optional `--top` (a number, default 2). It reads the file (each line
is `LEVEL service message`), keeps the lines of that level, and prints
how many there were, then the `--top` services with the most such lines
(highest first, ties by name) as `service: count`. The log file is
written for you in a temporary folder. Call `main` with two different
argument lists.

**Expected output:**

```text
ERROR lines: 5
db: 2
web: 2
INFO lines: 2
db: 1
```

<details><summary><b>Solution</b></summary>

```python
import argparse
from collections import Counter
def main(argv):
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--level", choices=["INFO", "WARN", "ERROR"], default="ERROR")
    parser.add_argument("--top", type=int, default=2)
    args = parser.parse_args(argv)
    counts = Counter()
    for line in Path(args.path).read_text().splitlines():
        level, service, _ = line.split(maxsplit=2)
        if level == args.level:
            counts[service] += 1
    print(args.level, "lines:", sum(counts.values()))
    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    for service, count in ranked[:args.top]:
        print(service + ": " + str(count))
main([str(log_path)])
main([str(log_path), "--level", "INFO", "--top", "1"])
folder.cleanup()
```

</details>

In [ ]:
import tempfile
from pathlib import Path
log_text = """ERROR db timeout
INFO web started
ERROR web 500
WARN db slow
ERROR db refused
ERROR cache miss
ERROR web 502
INFO db ok
"""
folder = tempfile.TemporaryDirectory()
log_path = Path(folder.name, "app.log")
log_path.write_text(log_text)

# your code here


### Q17 · Puzzle

Find the **big spenders**: customers whose orders add up to more than 50,
using the database to do the adding (`GROUP BY` and `HAVING`, with a
placeholder for the limit). Print `customer total` for each, biggest
total first.

**Expected output:**

```text
cy 70
ana 55
```

<details><summary><b>Solution</b></summary>

```python
query = """SELECT customer, SUM(amount) AS total FROM orders
         GROUP BY customer HAVING total > ? ORDER BY total DESC"""
for customer, total in conn.execute(query, (limit,)):
    print(customer, total)
```

</details>

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE orders (customer TEXT, amount INTEGER)")
conn.executemany("INSERT INTO orders VALUES (?, ?)", [
    ("ana", 30), ("ben", 12), ("ana", 25), ("cy", 70), ("ben", 15), ("dee", 5), ("ben", 9)])
limit = 50

# your code here


### Q18 · Puzzle

Back up and verify. Build a folder `project` with a few files (one in a
sub-folder), pack it with `shutil.make_archive` as a zip, unpack the
archive into a different folder with `shutil.unpack_archive`, and prove
the copy is identical: print `True` if both trees have the same sorted
relative file paths **and** the same SHA-256 hash for every file,
otherwise `False`. Also print the number of files.

**Expected output:**

```text
True
3
```

<details><summary><b>Solution</b></summary>

```python
import hashlib
import shutil
def fingerprint(root):
    result = {}
    for path in sorted(Path(root).rglob("*")):
        if path.is_file():
            result[path.relative_to(root).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
    return result
with tempfile.TemporaryDirectory() as folder:
    source = Path(folder, "project")
    for name, text in files.items():
        target = source / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)
    archive = shutil.make_archive(str(Path(folder, "backup")), "zip", source)
    restored = Path(folder, "restored")
    shutil.unpack_archive(archive, restored)
    print(fingerprint(source) == fingerprint(restored))
    print(len(fingerprint(restored)))
```

</details>

In [ ]:
import tempfile
from pathlib import Path
files = {"readme.txt": "hello", "src/app.py": "print(1)", "src/util.py": "x = 2"}

# your code here


### Q19 · Puzzle

Schema migrations. A database remembers its version in `PRAGMA
user_version`. Write `migrate(conn)` that applies, in order, only the
migrations in `migrations` newer than the stored version, and after each
one stores the new version number. Run it twice (the second run must do
nothing), printing the version and the list of tables (sorted) after
each run.

**Expected output:**

```text
3 ['posts', 'users']
3 ['posts', 'users']
```

<details><summary><b>Solution</b></summary>

```python
def migrate(conn):
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    for number, sql in enumerate(migrations, start=1):
        if number > version:
            with conn:
                conn.execute(sql)
                conn.execute(f"PRAGMA user_version = {number}")
def report(conn):
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name").fetchall()
    print(version, [name for (name,) in tables])
migrate(conn)
report(conn)
migrate(conn)
report(conn)
```

</details>

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
migrations = [
    "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)",
    "ALTER TABLE users ADD COLUMN email TEXT",
    "CREATE TABLE posts (id INTEGER PRIMARY KEY, user_id INTEGER, body TEXT)",
]

# your code here


---

## Q20 · Story project: Expense Tracker CLI

You are building a small command-line expense tracker for a friend who
keeps losing receipts. Everything is driven the way a real tool is: an
argument parser, a database, settings from a TOML file and the
environment, logging, and a final check that runs the report in a child
process. The database lives in memory and the files in a temporary folder,
so you can run it as often as you like. Each step builds on the one
before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
import tempfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
config_path = Path(workdir.name, "tracker.toml")
config_path.write_text("[budget]\nmonthly_cents = 20000\n\n[report]\ntop = 2\n")
environment = {"TRACKER_TOP": "3"}      # stands in for os.environ


### Step 1

Create the database: connect to `":memory:"` and make an `expenses`
table (`id` integer primary key, `day` text, `category` text,
`cents` integer, `note` text), with rows readable by column name.
Write `add_expense(conn, day, category, cents, note)` using
placeholders (inside a transaction). Add the four expenses in the
solution and print how many rows the table holds.

**Expected output:**

```text
4
```

<details><summary><b>Solution</b></summary>

```python
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("CREATE TABLE expenses (id INTEGER PRIMARY KEY, day TEXT, category TEXT, cents INTEGER, note TEXT)")
def add_expense(conn, day, category, cents, note):
    with conn:
        conn.execute("INSERT INTO expenses (day, category, cents, note) VALUES (?, ?, ?, ?)",
                     (day, category, cents, note))
add_expense(conn, "2025-03-01", "food", 1250, "lunch")
add_expense(conn, "2025-03-02", "travel", 8400, "train")
add_expense(conn, "2025-03-03", "food", 3100, "dinner; 'friends'")
add_expense(conn, "2025-03-04", "fun", 4500, "cinema")
print(conn.execute("SELECT COUNT(*) FROM expenses").fetchone()[0])
```

</details>

In [ ]:
# your code here


### Step 2

Build `make_parser()` with two sub-commands: `add DAY CATEGORY EUROS
NOTE` (EUROS is a decimal number) and `report` (with an optional
`--top`, a number). Parse `["add", "2025-03-05", "food", "12.50",
"snack"]` and `["report", "--top", "2"]` and print `command` and the
rest of each result's values (for `add` the four values with euros
converted to whole cents; for `report` the top).

**Expected output:**

```text
add 2025-03-05 food 1250 snack
report 2
```

<details><summary><b>Solution</b></summary>

```python
import argparse
def make_parser():
    parser = argparse.ArgumentParser(prog="tracker")
    sub = parser.add_subparsers(dest="command", required=True)
    add = sub.add_parser("add")
    add.add_argument("day")
    add.add_argument("category")
    add.add_argument("euros", type=float)
    add.add_argument("note")
    report = sub.add_parser("report")
    report.add_argument("--top", type=int)
    return parser
parser = make_parser()
args = parser.parse_args(["add", "2025-03-05", "food", "12.50", "snack"])
print(args.command, args.day, args.category, round(args.euros * 100), args.note)
args = parser.parse_args(["report", "--top", "2"])
print(args.command, args.top)
```

</details>

In [ ]:
# your code here


### Step 3

Load the settings: read the TOML file at `config_path` and take
`top` from `[report]` and the monthly budget from `[budget]`; if
`TRACKER_TOP` is in `environment`, it overrides the file's `top`
(convert to a number). Print the budget in cents and the final
`top`.

**Expected output:**

```text
budget: 20000
top: 3
```

<details><summary><b>Solution</b></summary>

```python
import tomllib
settings = tomllib.loads(config_path.read_text())
budget_cents = settings["budget"]["monthly_cents"]
top = settings["report"]["top"]
if "TRACKER_TOP" in environment:
    top = int(environment["TRACKER_TOP"])
print("budget:", budget_cents)
print("top:", top)
```

</details>

In [ ]:
# your code here


### Step 4

Write `run(argv)` that parses `argv` with your parser and does the
work. For `add` it adds the expense and logs `added <note>` at
INFO level. For `report` it prints the total spent in cents, the
categories with their totals (highest total first, at most `--top`
of them, or the settings' `top` if not given) as `category: cents`,
and logs a `WARNING` `over budget` when the total is above the
budget. Set up logging to stdout with the format `LEVEL message`.
Run the three argument lists in the solution: add a 2025-03-05
`fun` expense of 99.99 euros called `concert`, then `report`, then
`report --top 1`.

**Expected output:**

```text
INFO added concert
total: 27249
fun: 14499
travel: 8400
food: 4350
WARNING over budget
total: 27249
fun: 14499
WARNING over budget
```

<details><summary><b>Solution</b></summary>

```python
import logging
import sys
logging.basicConfig(stream=sys.stdout, force=True, level=logging.INFO, format="%(levelname)s %(message)s")
def run(argv):
    args = parser.parse_args(argv)
    if args.command == "add":
        add_expense(conn, args.day, args.category, round(args.euros * 100), args.note)
        logging.info("added %s", args.note)
    else:
        limit = args.top or top
        total = conn.execute("SELECT SUM(cents) FROM expenses").fetchone()[0]
        print("total:", total)
        rows = conn.execute("SELECT category, SUM(cents) AS spent FROM expenses "
                            "GROUP BY category ORDER BY spent DESC, category LIMIT ?", (limit,))
        for row in rows:
            print(row["category"] + ": " + str(row["spent"]))
        if total > budget_cents:
            logging.warning("over budget")
run(["add", "2025-03-05", "fun", "99.99", "concert"])
run(["report"])
run(["report", "--top", "1"])
```

</details>

In [ ]:
# your code here


### Step 5

Export and verify with a child process. Write all expenses (ordered
by id) as CSV lines `day,category,cents` into `expenses.csv`
inside `workdir`, put that file into a zip called `export.zip` in
the same folder, and then run a child Python process (the same
interpreter) with `-c` code that opens the zip, counts the lines
of `expenses.csv` and prints `lines in zip: N`. Pass the zip's path
as a separate argument, never inside a shell string. Print what the
child printed, then clean up the temporary folder.

**Expected output:**

```text
lines in zip: 5
```

<details><summary><b>Solution</b></summary>

```python
import subprocess
import zipfile
csv_path = Path(workdir.name, "expenses.csv")
lines = [f"{row['day']},{row['category']},{row['cents']}"
         for row in conn.execute("SELECT * FROM expenses ORDER BY id")]
csv_path.write_text("\n".join(lines) + "\n")
zip_path = Path(workdir.name, "export.zip")
with zipfile.ZipFile(zip_path, "w") as archive:
    archive.write(csv_path, arcname="expenses.csv")
child_code = ("import sys, zipfile\n"
              "with zipfile.ZipFile(sys.argv[1]) as z:\n"
              "    print('lines in zip:', len(z.read('expenses.csv').decode().splitlines()))")
result = subprocess.run([sys.executable, "-c", child_code, str(zip_path)],
                        capture_output=True, text=True, check=True)
print(result.stdout.strip())
workdir.cleanup()
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB32 · Advanced Capstone** in `advanced/`.